In [28]:
# Importing Libraries
import pandas as pd
from nltk import pos_tag
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
import string
import json

In [29]:
with open('../lexicon.json', 'r') as f:
    lexicon = json.load(f)
print(f'Lexicon loaded with length: {len(lexicon)}')

Lexicon loaded with length: 485092


In [30]:
# Loading the new document dataset
df = pd.read_csv('../Dataset/new_documents.csv')

In [31]:
# Printing the shape and head of the dataset
print(df.shape, '\n')
df.head()

(3, 10) 



,title,text,url,authors,timestamp,tags,text_length,title_length,num_tags,num_authors
0,ML is the future,Machine learning is the future of technology. ...,No url,"['Abdul Rehman', 'Tahir']",2020-07-01 00:00:00,"['ML', 'AI', 'Technology']",33,4,3,2
1,Abdul Rehman Article,This is a sample article written by Abdul Rehm...,No url,"['Abdul Rehman', 'Tahir']",2020-07-01 00:00:00,"['Test', 'Sample']",10,3,2,2
2,xxxxxxxxxxx,lksdajf jlksajf s jlasj f;llj kjsdal;jf sdljkk...,No url,"['Abdul Rehman', 'Tahir']",2020-07-01 00:00:00,"['Test', 'Sample']",8,1,2,2


In [32]:
# Creating the list of stopwords to be removed from the text
stop_words = set(stopwords.words('english'))

# Initializing the WordNetLemmatizer and SpellChecker
lemmatizer = WordNetLemmatizer()

<hr>

In [33]:
# Function to convert NLTK POS tags to WordNet POS tags
def get_wordnet_pos(tag):
    if tag.startswith('J'):  # Adjective
        return wordnet.ADJ
    elif tag.startswith('V'):  # Verb
        return wordnet.VERB
    elif tag.startswith('N'):  # Noun
        return wordnet.NOUN
    elif tag.startswith('R'):  # Adverb
        return wordnet.ADV
    else:
        return None  # Other POS

In [34]:
# Function to preprocess the title and text of the documents
def preprocess(title_text_pairs, doc_ids, lexicon):
    forward_index = {}
    
    for (title, text), doc_id in zip(title_text_pairs, doc_ids): 
        # Tokenization and normalization
        title_tokens = word_tokenize(title.lower())
        text_tokens = word_tokenize(text.lower())

        # Lemmatization and stopword removal
        title_tokens = [
            lemmatizer.lemmatize(word, pos=get_wordnet_pos(pos_tag([word])[0][1]) or 'n')
            for word in title_tokens if word.isalpha()
            and word not in stop_words 
            and word not in string.punctuation
        ]
        
        text_tokens = [
            lemmatizer.lemmatize(word, pos=get_wordnet_pos(pos_tag([word])[0][1]) or 'n')
            for word in text_tokens if word.isalpha()
            and word not in stop_words 
            and word not in string.punctuation
        ]

        # Mapping words to word IDs
        token_ids_title = []
        for word in title_tokens:
            if word not in lexicon:
                lexicon[word] = len(lexicon)  # Add new word to lexicon with unique ID
            token_ids_title.append(lexicon[word])

        token_ids_text = []
        for word in text_tokens:
            if word not in lexicon:
                lexicon[word] = len(lexicon)  # Add new word to lexicon with unique ID
            token_ids_text.append(lexicon[word])

        # Updating forward index with title and text word IDs
        forward_index[doc_id] = {
            'title': token_ids_title,
            'text': token_ids_text
        }

    return forward_index, lexicon

In [35]:
# Initializing the forward index
forward_index = {}

# Process the single document (there's no need for batch processing)
title = df['title'].iloc[-1]  
text = df['text'].iloc[-1]    

starting_doc_id = 192361
new_doc_id = starting_doc_id + len(df)
doc_id = f"doc{new_doc_id}"


# Preprocess the title and text of the document
forward_index, lexicon = preprocess([(title, text)], [doc_id], lexicon)


In [36]:
# Saving the updated lexicon to json file
with open('../lexicon.json', 'w') as f:
    json.dump(lexicon, f, indent=4)
print(f'Lexicon updated with new word(s). Length: {len(lexicon)}')

Lexicon updated with new word(s). Length: 485098


In [38]:
# Save the new forward index to the JSON file
with open('../new_forward_index.json', 'w') as f:
    json.dump(forward_index, f, indent=4)
print('Forward index saved to new_forward_index.json. Length:', len(forward_index))

Forward index saved to new_forward_index.json. Length: 1


<hr>